# Домашнее задание 2 — Planning Agent с ревизией + HITL паттерн

## Описание

Реализуйте двух агентов:
1. **Planning Agent с ревизией** — агент, который строит план, выполняет его и пересматривает при необходимости
2. **HITL-агент с опасными операциями** — Selective Human-in-the-Loop

---

## 📊 Оценка ДЗ (5-балльная шкала)

| Критерий | Баллы |
|---|---|
| Реализован **Planning Agent с ревизией** + **HITL паттерн** | **3** |
| Агент корректно отрабатывает **все тест-кейсы** | **2** |
| **Итого максимум** | **5** |

---
## 0. Подготовка

In [1]:
import warnings
import math

warnings.filterwarnings("ignore")

import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import settings
from openai import OpenAI

In [2]:
client = OpenAI(
    base_url="http://127.0.0.1:8080",
    api_key=settings.polza_ai_api_key,
)
MODEL = "deepseek/deepseek-chat-v3.1"

---
## 0.1 База данных городов и инструменты

Код из практики — необходим для выполнения заданий.

In [3]:
CITY_DATABASE = {
    "Москва": {
        "population": 12_600_000,
        "area_km2": 2561,
        "country": "Россия",
        "founded": 1147,
    },
    "Санкт-Петербург": {
        "population": 5_600_000,
        "area_km2": 1439,
        "country": "Россия",
        "founded": 1703,
    },
    "Новосибирск": {
        "population": 1_630_000,
        "area_km2": 505,
        "country": "Россия",
        "founded": 1893,
    },
    "Екатеринбург": {
        "population": 1_540_000,
        "area_km2": 468,
        "country": "Россия",
        "founded": 1723,
    },
    "Казань": {
        "population": 1_310_000,
        "area_km2": 614,
        "country": "Россия",
        "founded": 1005,
    },
    "Токио": {
        "population": 13_960_000,
        "area_km2": 2194,
        "country": "Япония",
        "founded": 1457,
    },
    "Пекин": {
        "population": 21_540_000,
        "area_km2": 16411,
        "country": "Китай",
        "founded": 1045,
    },
    "Нью-Йорк": {
        "population": 8_336_000,
        "area_km2": 783,
        "country": "США",
        "founded": 1624,
    },
    "Лондон": {
        "population": 8_982_000,
        "area_km2": 1572,
        "country": "Великобритания",
        "founded": 43,
    },
    "Париж": {
        "population": 2_161_000,
        "area_km2": 105,
        "country": "Франция",
        "founded": -250,
    },
}

---
## Задание: HITL-агент с опасными операциями

Реализуйте агента с **Human-in-the-Loop** для базы данных городов, в которой есть **опасные** операции.

### Шаг 1: Добавьте «опасные» инструменты

| Инструмент | Что делает | Опасный? |
|-----------|-----------|---------|
| `search_city` | Читает данные | Нет |
| `list_cities` | Читает список | Нет |
| `calculate` | Вычисляет | Нет |
| `add_city(name, population, area_km2, country, founded)` | Добавляет город в базу | **Да** |
| `update_city(city_name, field, value)` | Изменяет поле города | **Да** |
| `delete_city(city_name)` | Удаляет город из базы | **Да** |

### Шаг 2: Реализуйте Selective HITL

Напишите `run_hitl_agent(question, dangerous_tools, auto_approve=False)`:
- **Безопасные** инструменты (`search_city`, `list_cities`, `calculate`) выполняются автоматически
- **Опасные** инструменты (`add_city`, `update_city`, `delete_city`) требуют подтверждения от человека
- Если человек отклонил — агент получает `{"error": "Действие отклонено пользователем"}` и адаптируется

### Шаг 3: Проверьте на сценарии

Протестируйте агента на запросе:

> *«Добавь в базу город Владивосток (население 600000, площадь 331 км², Россия, основан 1860). Затем найди город России с наименьшим населением.»*

Ожидаемое поведение:
1. Агент вызывает `add_city` → человек одобряет (или отклоняет!)
2. Агент вызывает `list_cities(country="Россия")` → автоматически
3. Агент ищет данные по городам → автоматически
4. Агент даёт ответ

In [4]:
def search_city(city_name: str) -> dict:
    info = CITY_DATABASE.get(city_name)
    if info:
        return {"status": "found", "city": city_name, **info}
    return {
        "status": "not_found",
        "city": city_name,
        "error": f"Город '{city_name}' не найден в базе",
    }


def list_cities(country: str | None = None) -> dict:
    if country:
        cities = [name for name, data in CITY_DATABASE.items() if data["country"] == country]
    else:
        cities = list(CITY_DATABASE.keys())
    return {"cities": cities, "count": len(cities)}


def calculate(expression: str) -> dict:
    namespace = {
        "__builtins__": {},
        "math": math,
        "pi": math.pi,
        "e": math.e,
        "sqrt": math.sqrt,
        "abs": abs,
        "round": round,
        "max": max,
        "min": min,
    }
    try:
        result = eval(expression, namespace)
        return {"expression": expression, "result": result}
    except Exception as ex:
        return {"expression": expression, "error": str(ex)}


TOOL_FUNCTIONS = {
    "search_city": search_city,
    "list_cities": list_cities,
    "calculate": calculate,
}

print("Инструменты готовы:", list(TOOL_FUNCTIONS.keys()))

Инструменты готовы: ['search_city', 'list_cities', 'calculate']


In [5]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "search_city",
            "description": "Получить информацию о городе: население (population), площадь (area_km2), страну (country), год основания (founded)",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {
                        "type": "string",
                        "description": "Название города на русском языке",
                    }
                },
                "required": ["city_name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_cities",
            "description": "Получить список доступных городов в базе. Можно отфильтровать по названию страны",
            "parameters": {
                "type": "object",
                "properties": {
                    "country": {
                        "type": "string",
                        "description": "Название страны для фильтрации (необязательно)",
                    }
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Вычислить математическое выражение. Поддерживает: +, -, *, /, **, sqrt(), round(), max(), min(), pi",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Математическое выражение, например '12600000 / 2561' или 'round(3.14159, 2)'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
]

In [6]:
# ============================================================
#  Шаг 1: Реализуйте «опасные» инструменты
# ============================================================
# Подсказка: CITY_DATABASE — это обычный dict.
# Мутировать его из функции можно напрямую


def add_city(name: str, population: int, area_km2: float, country: str, founded: int) -> dict:
    """Добавить новый город в базу данных."""
    # TODO: проверить что города нет в CITY_DATABASE
    # Если есть — вернуть {"error": "Город уже существует"}
    # Если нет — добавить: CITY_DATABASE[name] = {"population": ..., "area_km2": ..., ...}
    # Вернуть {"status": "added", "city": name}
    # ...
    if not CITY_DATABASE.get(name):
        CITY_DATABASE[name] = {
            "population": population,
            "area_km2": area_km2,
            "country": country,
            "founded": founded,
        }
        return {"status": "added", "city": name}
    else:
        return {"error" : "Город уже существует"}



def update_city(city_name: str, field: str, value) -> dict:
    """Обновить поле существующего города."""
    # TODO: проверить что город есть в CITY_DATABASE
    # Проверить что поле валидное (population, area_km2, country, founded)
    # Обновить: CITY_DATABASE[city_name][field] = value
    # Вернуть {"status": "updated", "city": city_name, "field": field, "new_value": value}
    # ...
    if CITY_DATABASE.get(city_name):
        if field in ("population", "area_km2", "country", "founded"):
            CITY_DATABASE[city_name][field] = value
            return {
                "status" : "update",
                "city" : city_name,
                "field" : field,
                "new_value" : value
            }
        else: 
            return {"error" : "Поле не валидное"}
    else:
        return {"error" : "Такого города нет"}



def delete_city(city_name: str) -> dict:
    """Удалить город из базы данных."""
    # TODO: проверить что город есть в CITY_DATABASE
    # Удалить: del CITY_DATABASE[city_name]
    # Вернуть {"status": "deleted", "city": city_name}
    # ...

    if CITY_DATABASE.get(city_name):
        del CITY_DATABASE[city_name]
        return {"status": "deleted", "city": city_name}
    else: return {"error" : "Такого города нет"}

# TODO: добавьте описания инструментов
# TODO: добавьте функции в маппинг

DANGEROUS_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "add_city",
            "description": "Добавить новый город в базу данных. Если город уже существует, вернёт ошибку.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Название города (уникальное)"
                    },
                    "population": {
                        "type": "integer",
                        "description": "Количество населения (целое число)"
                    },
                    "area_km2": {
                        "type": "number",
                        "description": "Площадь города в км² (дробное число)"
                    },
                    "country": {
                        "type": "string",
                        "description": "Страна, в которой находится город"
                    },
                    "founded": {
                        "type": "integer",
                        "description": "Год основания города (целое число)"
                    }
                },
                "required": ["name", "population", "area_km2", "country", "founded"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "update_city",
            "description": "Обновить одно из полей существующего города: population, area_km2, country, founded.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {
                        "type": "string",
                        "description": "Название города, который нужно обновить"
                    },
                    "field": {
                        "type": "string",
                        "enum": ["population", "area_km2", "country", "founded"],
                        "description": "Поле для обновления"
                    },
                    "value": {
                        "type": "number",
                        "description": "Новое значение. Для country ожидается строка, но схема допускает number; корректность проверяется в коде."
                    }
                },
                "required": ["city_name", "field", "value"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "delete_city",
            "description": "Удалить город из базы данных.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {
                        "type": "string",
                        "description": "Название города для удаления"
                    }
                },
                "required": ["city_name"]
            }
        }
    }
]



DANGEROUS_TOOL_FUNCTIONS = {
    # TODO: маппинг имя → функция для опасных инструментов
    "add_city": add_city,
    "update_city": update_city,
    "delete_city": delete_city,
}

In [7]:
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
import json

In [ ]:
@dataclass
class AgentStep:
    step_number: int
    tool_calls: list[dict] = field(default_factory=list)
    observation: list[dict] = field(default_factory=list)
    final_answer: str | None = None


@dataclass
class AgentTrace:
    question: str
    steps: list[AgentStep] = field(default_factory=list)
    total_steps: int = 0
    final_answer: str = ''

    def summary(self) -> str:
            lines = [f"Вопрос: {self.question}", f"Шагов: {self.total_steps}", ""]
            for step in self.steps:
                lines.append(f"--- Шаг {step.step_number} ---")
                for tool_call in step.tool_calls:
                    lines.append(f"  Вызов: {tool_call['name']}({tool_call['args']})")
                for observation in step.observations:
                    lines.append(f"  Результат: {observation}")
                if step.final_answer:
                    lines.append(f"  Ответ: {step.final_answer}")
            lines.append(f"\nФинальный ответ: {self.final_answer}")
            return "\n".join(lines)

    

In [9]:
# ============================================================
# Шаг 2: Реализуйте Selective HITL-агент
# ============================================================

# Множество инструментов, требующих одобрения человека
TOOLS_REQUIRING_APPROVAL = {"add_city", "update_city", "delete_city"}


SYSTEM_PROMPT = """Ты аналитик данных. У тебя есть доступ к базе данных городов мира.

Правила:
- Всегда используй инструменты для получения данных — не придумывай числа
- Если нужно вычислить что-то — используй calculate
- Когда собрал достаточно данных — дай чёткий, структурированный ответ
- Если данных нет — честно скажи об этом"""

def run_hitl_agent(
    question: str,
    tools_schema: list | None = None,
    tool_functions: dict | None = None,
    dangerous_tools: set | None = None,
    max_steps: int = 10,
    console_print: bool = True,
    temperature: float = 0.15,
    auto_approve: bool = False,
) -> dict:
    """
    Агент с Human-in-the-Loop (selective approval).

    - Безопасные инструменты выполняются автоматически
    - Опасные инструменты требуют подтверждения от человека
    - При отклонении — агент получает ошибку и адаптируется
    """
    # TODO: реализуйте агентный цикл
    # Подсказка: если func_name in dangerous_tools → спрашивайте input("Разрешить? (y/n): ")
    # Если не одобрено — result = {"error": "Действие отклонено пользователем"}
    # ...э


    tool_functions = tool_functions or TOOL_FUNCTIONS
    dangerous_tools = dangerous_tools or DANGEROUS_TOOL_FUNCTIONS
    tools_schema = tools_schema or TOOLS_SCHEMA

    trace = AgentTrace(question=question)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    if console_print:
        print(f"[State] Вопрос: {question}")

    for step_num in range(1, max_steps + 1):
        if console_print:
            print(f"\n{'=' * 50}")
            print(f"--- Шаг {step_num} ---")

        # Decision: LLM решает — вызвать инструмент или дать ответ
        if console_print:
            print(f"  [Decision] LLM анализирует состояние...")

        response = client.chat.completions.create(
            model = MODEL,
            messages=messages,
            tools=tools_schema,
            temperature=temperature,
        )
        msg = response.choices[0].message
        messages.append(msg)
        step = AgentStep(step_number=step_num)

        if not msg.tool_calls:
            step.final_answer = msg.content
            trace.steps.append(step)
            trace.final_answer = msg.content or ""
            trace.total_steps = step_num
            if console_print:
                print(f"  [Decision] → Финальный ответ (нет tool_calls)")
                print(f"  [Result] {msg.content}")
            break


        # if msg.tool_calls:
        step_result = []
        for tc in msg.tool_calls:
            name = tc.function.name
            # args = tc.function.arguments
            args = json.loads(tc.function.arguments)

            if console_print:
                print(f"  [Action] {name}({args})")

            if name in dangerous_tools:
                if auto_approve:
                    func = dangerous_tools.get(name)
                    result = func(**args)

                    if console_print:
                        print(f"  [Observation] {result}")

                    step.tool_calls.append({"name": name, "args": args})
                    step.observation.append(result)
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tc.id,
                            "content": json.dumps(result, ensure_ascii=False),
                        }
                    )
                else:

                    approved = input(f"Выполнить функцию: {name}? (y/N)").strip().lower() == 'y'


                    if approved:
                        func = dangerous_tools.get(name)
                        result = func(**args)

                        if console_print:
                            print(f"  [Observation] {result}")

                        step.tool_calls.append({"name": name, "args": args})
                        step.observation.append(result)
                        messages.append(
                            {
                                "role": "tool",
                                "tool_call_id": tc.id,
                                "content": json.dumps(result, ensure_ascii=False),
                            }
                        )

                    else:
                        # step.final_answer = msg.content
                        # trace.steps.append(step)
                        # trace.final_answer = msg.content or ""
                        # trace.total_steps = step_num
                        # if console_print:
                        #     print(f"  [Decision] → Финальный ответ (нет tool_calls)")
                        #     print(f"  [Result] {msg.content}")
                        # break
                        result = {
                            "error": "Действие отклонено пользователем"
                        }
                        if console_print:
                            print(f"  [HITL] Действие отклонено")
                            print(f"  [Observation] {result}")

                        step.tool_calls.append({"name": name, "args": args})
                        step.observation.append(result)

                        messages.append({
                            "role": tool,
                            "tool_call_id": tc.id,
                            "content": json.dumps(result, ensure_ascii=False)
                        })

            else:
                func = tool_functions.get(name)
                result = func(**args)

                if console_print:
                    print(f"  [Observation] {result}")

                step.tool_calls.append({"name": name, "args": args})
                step.observation.append(result)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": json.dumps(result, ensure_ascii=False),
                    }
                )
            trace.steps.append(step)
        else:
            trace.total_steps = max_steps
            trace.final_answer = "Достигнут лимит шагов"
    
    return trace

In [10]:
# ============================================================
# Тест-кейсы для проверки работы агента
# ============================================================

print("\n\nКейс 1: Только безопасные операции — поиск информации")
hitl_result = run_hitl_agent(
    "Добавь в базу город Владивосток (население 600000, площадь 331 км², Россия, основан 1860). "
    "Затем найди город России с наименьшим населением.",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)

print("\n\nКейс 2: Удаление города (опасная операция)")
hitl_result = run_hitl_agent(
    "Удали Париж из базы данных.",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)

print("\n\nКейс 3: Обновление поля города (опасная операция)")
hitl_result = run_hitl_agent(
    "Обнови население Москвы на 13000000.",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)

print("\n\nКейс 4: Безопасный расчёт — плотность населения")
hitl_result = run_hitl_agent(
    "Вычисли плотность населения (чел/км²) для Пекина.",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)

print("\n\nКейс 5: Добавление + удаление (две опасные операции подряд)")
hitl_result = run_hitl_agent(
    "Добавь город Сочи (население 440000, площадь 176 км², Россия, основан 1838), "
    "а затем удали Новосибирск из базы.",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)

print("\n\nКейс 6: Список всех городов (только безопасное)")
hitl_result = run_hitl_agent(
    "Перечисли все города Россиии из базы данных.",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)

print("\n\nКейс 7: Добавление уже существующего города (ожидаем ошибку от инструмента)")
hitl_result = run_hitl_agent(
    "Добавь город Казань (население 1310000, площадь 614 км², Россия, основан 1005).",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)

print("\n\nКейс 8: Сравнение двух городов (безопасные вызовы + вычисление)")
hitl_result = run_hitl_agent(
    "Сравни Лондон и Нью-Йорк: у какого города больше плотность населения?",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)

print("\n\nКейс 9: Обновление нескольких полей (несколько опасных вызовов)")
hitl_result = run_hitl_agent(
    "Обнови данные Екатеринбурга: население — 1600000, площадь — 490 км².",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)

print("\n\nКейс 10: Смешанный сценарий — безопасный поиск + опасное добавление + расчёт")
hitl_result = run_hitl_agent(
    "Найди самый старый город в базе. Затем добавь город Дубай "
    "(население 3500000, площадь 1588 км², ОАЭ, основан 1833) "
    "и вычисли, во сколько раз население Дубая больше населения Казани.",
    tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
    tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
    dangerous_tools=DANGEROUS_TOOL_FUNCTIONS,
    auto_approve=True,
)



Кейс 1: Только безопасные операции — поиск информации
[State] Вопрос: Добавь в базу город Владивосток (население 600000, площадь 331 км², Россия, основан 1860). Затем найди город России с наименьшим населением.

--- Шаг 1 ---
  [Decision] LLM анализирует состояние...
  [Action] add_city({'name': 'Владивосток', 'population': 600000, 'area_km2': 331, 'country': 'Россия', 'founded': 1860})
  [Observation] {'status': 'added', 'city': 'Владивосток'}

--- Шаг 2 ---
  [Decision] LLM анализирует состояние...
  [Action] list_cities({'country': 'Россия'})
  [Observation] {'cities': ['Москва', 'Санкт-Петербург', 'Новосибирск', 'Екатеринбург', 'Казань', 'Владивосток'], 'count': 6}

--- Шаг 3 ---
  [Decision] LLM анализирует состояние...
  [Action] search_city({'city_name': 'Москва'})
  [Observation] {'status': 'found', 'city': 'Москва', 'population': 12600000, 'area_km2': 2561, 'country': 'Россия', 'founded': 1147}
  [Action] search_city({'city_name': 'Санкт-Петербург'})
  [Observation] {'status